# Reference Design - Four multiplexed readout

This design was taken from the quantum metals reference designs. This design  was altered to remove the four transmon devices and include only the resonators at their respective lengths.

In [1]:
import qiskit_metal as qm
from qiskit_metal import Dict, designs, MetalGUI
from qiskit_metal.qlibrary.qubits.transmon_pocket import TransmonPocket
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond

design = designs.DesignPlanar()
design.overwrite_enabled = True

In [2]:
def feed(a,ap,b,bp,name):
    '''
    Auto-route a coplanar waveguide feedline segement between two pins

    Inputs:
    a       - 
    ap      -
    b       -
    bp      -
    name    - 

    Output:
    '''
    RoutePathfinder(
        design,
        name,
        options = dict(
            fillet ='90um',
            pin_inputs = Dict(
                start_pin = Dict(component=a, pin=ap), end_pin=Dict(component=b, pin=bp) 
            ),
        ),
    )


def readout(clt, q, name, length):
    '''
    Meandered lambbda/4 readout resonator: coupled-line tee --> qubi readout pad
    '''
    RouteMeander(
        design,
        name,
        options = dict(
            fillet ='90um',
            total_length=length,
            lead=Dict(start_straight = '100um', end_straight = '100um'),
            pin_inputs = Dict(
                start_pin = Dict(component=clt,pin='second_end'),
                end_pin=Dict(component=q,pin='read')
            )
        )
    )

## 1. Four Readout Tees
A shared ainglw feedline coupled to four readout tees.

In [3]:
xs = [-4.5, -1.5, 1.5, 4.5]
for i , x in enumerate(xs,1):
    CoupledLineTee(
        design,
        f"clt{i}",
        options = dict(
            pos_x = f"{x}mm",
            pos_y ="1.8mm",
            coupling_length = '350um',
            down_length = '300um',
            fillet = '90um',
            open_termination = False,
        ),
    )

## 2. One Shared Feedline through all four tees

In [4]:
LaunchpadWirebond(
    design,"LPin", options = dict(pos_x='-7.5mm',pos_y='1.8mm',orientation='0')
)
LaunchpadWirebond(
    design,'LPout', options = dict(pos_x='7.5mm',pos_y = '1.8mm',orientation='180')
)
feed('LPin','tie','clt1','prime_start','f0')

for i in range(1,4):
    feed(f'clt{i}','prime_end', f'clt{i+1}','prime_start',f'f{i}')

feed('clt4','prime_end','LPout','tie','f4')

## 3. Four frequency multiplexed readout resonators
Stepped length -> distinct readout frequencies on the one feedline

In [5]:
for i, L in zip(range(1,5),['6.8mm','7.0mm','7.2mm','7.4mm']):
    readout(f'clt{i}',f'Q{i}',f'read{i}',L)

01:55PM 41s WARNING [__init__]: Component Q1 does not exist. read1 has not been built. Please check your pin_input values.
01:55PM 41s WARNING [__init__]: Component Q2 does not exist. read2 has not been built. Please check your pin_input values.
01:55PM 41s WARNING [__init__]: Component Q3 does not exist. read3 has not been built. Please check your pin_input values.
01:55PM 41s WARNING [__init__]: Component Q4 does not exist. read4 has not been built. Please check your pin_input values.


In [6]:
design.components.keys()

['clt1', 'clt2', 'clt3', 'clt4', 'LPin', 'LPout', 'f0', 'f1', 'f2', 'f3', 'f4']

In [7]:
design = designs.DesignPlanar()
# ... build your components on `design` ...

gui = MetalGUI(design)
gui.rebuild()      # re-render after adding/editing components
gui.autoscale()     # fit view to design